In [ ]:
%reload_ext autoreload
%autoreload 2
%cd ~/erc-src/cuneiform-ocr-sign-alignment-worktree
%env PATH=$HOME/.local/bin:$PATH

import os
import cv2
import numpy as np
import torch
from dotenv import load_dotenv

from sign_alignment.detector import ModelConfig, TabletImageDetector
from sign_alignment.data_source import LocalDataSource
from sign_alignment.visualizer import ColorConfig

ANNOTATIONS_DIR = os.path.expanduser("~/erc-work-data/data-of-cuneiform-ocr-data/filtered_annotations")
CONFIG_FILE = "configs/detr.py"
CHECKPOINT_FILE = os.path.expanduser("~/erc-work-data/retrained_models/detr-173/epoch_1000.pth")
# # temporal change
# CHECKPOINT_FILE = os.path.expanduser("~/epoch_1000.pth")
# ANNOTATIONS_DIR = os.path.expanduser("~/filtered_annotations")
# # ---
SCORE_THRESHOLD = 0.0
OUTPUT_DIR = "alignment_results"
SAMPLE_LIMIT = 5

load_dotenv()
MONGODB_URI = os.getenv('MONGODB_URI', 'YOUR_MONGODB_URI')
CANONICAL_FEATURE_DIR = "~/erc-work-data/signs_alignment_data/precompute_feautures/"
if not MONGODB_URI or MONGODB_URI == 'YOUR_MONGODB_URI':
    raise ValueError("MONGODB_URI is required for Mongo-backed canonical sign images")



# --- show ipynb kernel connection info ---
import sys
from pathlib import Path

from ipykernel.connect import get_connection_file
from jupyter_core.paths import jupyter_runtime_dir

runtime_dir = Path(jupyter_runtime_dir()).expanduser()
kernel_file = Path(get_connection_file()).expanduser()
if not kernel_file.is_absolute():
    kernel_file = runtime_dir / kernel_file
kernel_file = kernel_file.resolve()
jupyter_bin = Path(sys.executable).with_name("jupyter")

print("runtime dir:", runtime_dir)
print("kernel file:", kernel_file)
print("connect cmd:", f"{jupyter_bin} console --existing {kernel_file}")

In [ ]:
from sign_alignment.pipeline import CropContext, Runner, VisOptions
from sign_alignment.data_source import EBLMongoCanonicalSource, PrototypeSource
from sign_alignment.dift_align import DiftAlignmentConfig, DiftRuntime

_DIFT_CHECKPOINT = os.path.expanduser("~/erc-src/ProtoSnap/weights/SD_with_prompt")
sign_source = EBLMongoCanonicalSource(MONGODB_URI)
proto_src = PrototypeSource()
dift = DiftRuntime(
    checkpoint=_DIFT_CHECKPOINT,
    feature_dir=CANONICAL_FEATURE_DIR,
    config=DiftAlignmentConfig(affine_probe_padding_ratio=0.1),
)

model_config = ModelConfig(
    config_file=CONFIG_FILE,
    checkpoint_file=CHECKPOINT_FILE,
    device='auto'
)

if 'tablet_detector' not in globals() or getattr(tablet_detector, 'model', None) is None:
    tablet_detector = TabletImageDetector(
        default_score_threshold=SCORE_THRESHOLD,
        model_config=model_config,
        keep_crops=True,
        is_crop_itself=False,
    )
else:
    
    print("Reusing existing tablet_detector instance.")

crop_context = CropContext(
    tablet_detector=tablet_detector,
    local_source=LocalDataSource(ANNOTATIONS_DIR),
    color_config=ColorConfig,
    output_dir=OUTPUT_DIR,
    img_idx=1,
    dift=dift,
    sign_source=proto_src,
)

vis = VisOptions(info=True, display=True, save=True)

runner = Runner(context=crop_context, vis=vis)


In [ ]:
from sign_alignment.data_source import PrototypeSource
src_proto = PrototypeSource()

img = src_proto.get("AN", 'Neo Assyrian')
import matplotlib.pyplot as plt
plt.imshow(img, cmap="gray", vmin=0, vmax=255)
plt.axis('off')
plt.show() # Try image without background

In [ ]:

# periods for selected fragments
check_periods = runner._fragments[0:20]

periods = []
for fragment_id in check_periods:
    fragment_data = runner.context.api_source.get_fragment_data(fragment_id)
    periods.append({
        "fragment": fragment_id,
        "period": (fragment_data or {}).get("script", {}).get("period", "<missing>"),
    })

periods

In [ ]:
import sign_alignment.pipeline as pp

runner.choose_sample(9)  # index 0 = NBC.4020, index 9 = HS.2086
# Load image, ground truth, and sign text from API in one step
# runner.choose_sample(name="YBC.12860")  # switch sample here
# runner.choose_sample(name="ND.5437")  # switch sample here
runner.run([pp.Step("Load data", pp.load_data, pp.vis_loaded_data)])


In [ ]:
# detect signs (full image + chosen exp_image crop)
runner.choose_crop(5)
runner.run([pp.Step("Detect signs", pp.detect_signs, pp.vis_detections)])

# Temporary experiment: crop by height/width ratios and detect again
from sign_alignment.detector import SingleImageDetector
from sign_alignment.tablet import SubTablet
from sign_alignment.visualizer import BboxVisualizer

crop_tablet = runner.context.state.crop_tablet
height_start_ratio, height_end_ratio = 0, 2 / 5
width_start_ratio, width_end_ratio = 0, 3 / 5
# height_start_ratio, height_end_ratio = 3/ 10, 4 / 10
# width_start_ratio, width_end_ratio = 2 / 5, 3 / 5

h, w = crop_tablet.shape
y1, y2 = int(h * height_start_ratio), int(h * height_end_ratio)
x1, x2 = int(w * width_start_ratio), int(w * width_end_ratio)
further_crop_tablet = SubTablet(
    img=crop_tablet.img[y1:y2, x1:x2].copy(),
    parent=crop_tablet,
    offset_in_parent=(x1, y1),
    mask=crop_tablet.mask[y1:y2, x1:x2].copy() if crop_tablet.mask is not None else None,
    name="further_crop_by_ratio",
)
further_det_boxes = SingleImageDetector(
    model=tablet_detector.model,
    default_score_threshold=SCORE_THRESHOLD,
).detect(further_crop_tablet)

print(f"Further crop: x={x1}:{x2}, y={y1}:{y2}, shape={further_crop_tablet.img.shape}")
print(f"Further crop detections: {len(further_det_boxes)}")
further_vis = BboxVisualizer(color=ColorConfig.DET_COLOR.value)
further_vis.draw_boxes(further_crop_tablet.img, further_det_boxes)
further_vis.display_result(vis_opt="draw")


In [ ]:
# transform GT boxes into sub-image coordinates and visualize
runner.run([pp.Step("Transform GT to crop", pp.transform_gt_to_crop, pp.vis_crop_ground_truth)])


In [ ]:
# compute average detection box dimensions
runner.run([pp.Step("Detection statistics", lambda _: None, pp.vis_detection_statistics)])

In [ ]:
# create detection and text sub-tablets
runner.run([pp.Step("Create box sets", pp.create_box_sets, pp.vis_box_sets)])

In [ ]:
# DBSCAN row detection on detection sub-tablet (also reports text sub-tablet rows)
runner.run([pp.Step("Detect rows", pp.detect_rows, pp.vis_detected_rows_info)])

In [ ]:
# DP row matching between detection and text sub-tablets
runner.run([pp.Step("Match rows", pp.match_rows, pp.vis_row_matches)])

In [ ]:
# visualize detection rows with D# / D#→R# labels
runner.run([pp.Step("Visualize detection rows", lambda _: None, pp.vis_detection_rows)])

In [ ]:
# within-row sign matching for each matched row pair
runner.run([pp.Step("Match signs", pp.match_signs_in_rows, pp.vis_sign_matches)])

In [ ]:
# align text rows onto detection rows using regression baselines
# result stored in aligned_boxes
runner.run([pp.Step("Align text rows", pp.align_text_rows, pp.vis_aligned_rows)])


In [ ]:
# use detection geometry directly and only correct matched labels from text alignment
# stored separately; the original coarse alignment remains the PSR optimizer input
runner.run([pp.Step(
    "Result without optimization",
    pp.create_result_without_optimization,
    pp.vis_result_without_optimization,
)])


In [ ]:
# build sign match info; draw text mapping, side-by-side composite, alignment diagnostic
runner.run([pp.Step("Build sign match info", pp.build_sign_match_info, pp.vis_sign_match_info)])

In [ ]:
# position offset analysis: coarse-aligned vs detection boxes
runner.run([pp.Step("Offset analysis", lambda _: None, pp.vis_offset_analysis)])


In [ ]:
# unload detector model from GPU to free VRAM before DIFT / PSR optimization
runner.run([pp.Step("Unload detector", pp.unload_detector)])  # optional

In [ ]:
# Source setup records the period; images and features are queried on demand

# runner.run([pp.Step("Setup source signs", pp.setup_source_signs, pp.vis_source_signs)])
runner.run([pp.Step("Setup source signs", pp.setup_source_signs)]) # skip vis now

In [ ]:
from sign_alignment.sign import Sign, SignResolver
resolver = SignResolver()

period = pp._source_period(runner.context)
features = runner.context.dift.get_sign_feature(resolver.from_name("AN"), period)

features.shape

In [ ]:
# DIFT sampling-grid scores on the crop tablet (includes sim_withoutbg heatmap)
runner.run([pp.Step("DIFT sampling grid scores", lambda _: None, pp.vis_dift_score_on_whole_tablet)])

In [ ]:
# calculate x1 y1 from center coordinates
s = runner.context.state
box_width = s.detections.avg_width
box_height = s.detections.avg_height
print(f"Box dimensions: {round(box_width)} x {round(box_height)}")

coordinates_to_check = [(34, 37), (41, 16), (28, 31),(15, 38), (8, 15)]
x_y_coords = [(y, x) for x, y in coordinates_to_check] # revert xy
first_cx, first_cy = 200, 200
step_x, step_y = 50, 50

x1s = [first_cx + step_x * dx - 0.5 * box_width for dx, dy in x_y_coords]
y1s = [first_cy + step_y * dy - 0.5 * box_height for dx, dy in x_y_coords]

print("Coordinates to check (x1, y1):")
for x1, y1 in zip(x1s, y1s):
    print(f"({round(x1)}, {round(y1)})")



In [ ]:
# Select a crop by x1, y1, width, height and compare it with a canonical sign using DIFT features, including sim_withoutbg
runner.run([pp.Step("Manual DIFT crop match", lambda _: None, pp.vis_manual_dift_crop_match)])

In [ ]:
# Visualize sim[i, j, :, :] for one prototype/source feature-grid point.
# Use the sliders to choose the source feature-grid point.
import cv2
import matplotlib.pyplot as plt
import torch
import ipywidgets as widgets
from IPython.display import clear_output, display

from sign_alignment.box import Box
from sign_alignment.dift_align import ImageView
from sign_alignment.sign import SignResolver

s = runner.context.state

sim_sign_name = "AN"
sim_x1, sim_y1 = 1790, 1720
sim_width, sim_height = 420, 279

# sim_x1, sim_y1 = 1020, 1720 # not AN box
# sim_width, sim_height = 420, 279

sim_box = Box(
    x1=sim_x1,
    y1=sim_y1,
    x2=sim_x1 + sim_width,
    y2=sim_y1 + sim_height,
    sign=SignResolver.from_name(sim_sign_name),
    tablet=s.crop_tablet,
)
crop_img = sim_box.crop_image()

source = runner.context.dift.source
if source is None:
    raise RuntimeError("DiftRuntime.source must be set.")
period = pp._source_period(runner.context)
source_img = source.get(sim_sign_name, period)
source_feature = runner.context.dift.get_sign_feature(sim_box.sign, period)
if source_img is None or source_feature is None:
    raise RuntimeError(f"No source image/feature for sign {sim_sign_name!r}.")

crop_feature = runner.context.dift.featurize_image(crop_img)
src = source_feature.detach().float()
dst = crop_feature.detach().float().to(device=src.device)
sim_hwhw = torch.einsum("cij,ckl->ijkl", src, dst).detach().cpu()

src_c, src_h, src_w = src.shape
dst_h, dst_w = dst.shape[-2:]
source_rgb = ImageView.from_any(source_img).as_rgb_numpy()
crop_rgb = ImageView.from_any(crop_img).as_rgb_numpy()
source_img_h, source_img_w = source_rgb.shape[:2]
crop_img_h, crop_img_w = crop_rgb.shape[:2]

print(
    f"sign={sim_sign_name!r}, crop=(x1={sim_x1}, y1={sim_y1}, "
    f"w={sim_width}, h={sim_height}), sim shape={tuple(sim_hwhw.shape)}"
)
print(
    f"source feature: C={src_c}, H={src_h}, W={src_w}; "
    f"target feature: H={dst_h}, W={dst_w}"
)

src_i_input = widgets.IntSlider(
    value=src_h // 2,
    min=0,
    max=src_h - 1,
    step=1,
    description="src i",
    continuous_update=False,
    readout=True,
    layout=widgets.Layout(width="420px"),
    style={"description_width": "48px"},
)
src_j_input = widgets.IntSlider(
    value=src_w // 2,
    min=0,
    max=src_w - 1,
    step=1,
    description="src j",
    continuous_update=False,
    readout=True,
    layout=widgets.Layout(width="420px"),
    style={"description_width": "48px"},
)
out = widgets.Output()

def _draw_sim_slice(i, j):
    with out:
        clear_output(wait=True)
        i = int(i)
        j = int(j)
        heat = sim_hwhw[i, j].numpy()
        heat_min = float(heat.min())
        heat_max = float(heat.max())
        heat_norm = (heat - heat_min) / max(heat_max - heat_min, 1e-6)
        heat_overlay = cv2.resize(
            heat_norm,
            (crop_img_w, crop_img_h),
            interpolation=cv2.INTER_CUBIC,
        )
        src_x = (j + 0.5) / src_w * source_img_w
        src_y = (i + 0.5) / src_h * source_img_h

        fig, axes = plt.subplots(1, 4, figsize=(18, 4.5), constrained_layout=True)
        axes[0].imshow(source_rgb)
        axes[0].scatter([src_x], [src_y], c="cyan", s=70, edgecolors="black", linewidths=1.2)
        axes[0].set_title(f"prototype/source grid ({i}, {j})")

        axes[1].imshow(crop_rgb)
        axes[1].set_title("target crop")

        im = axes[2].imshow(heat, cmap="magma")
        axes[2].set_title(f"sim[{i}, {j}, :, :]")
        fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

        axes[3].imshow(crop_rgb)
        axes[3].imshow(heat_overlay, cmap="magma", alpha=0.45, vmin=0.0, vmax=1.0)
        axes[3].set_title("target overlay")

        for ax in axes:
            ax.axis("off")
        display(fig)
        plt.close(fig)

def _redraw_on_change(_change):
    _draw_sim_slice(src_i_input.value, src_j_input.value)

src_i_input.observe(_redraw_on_change, names="value")
src_j_input.observe(_redraw_on_change, names="value")
display(widgets.HBox([src_i_input, src_j_input]), out)
_draw_sim_slice(src_i_input.value, src_j_input.value)


In [ ]:
print(sim_hwhw[:2].shape)

i = int(34)
j = int(32)


heat = sim_hwhw[i, j].numpy()
heat_min = float(heat.min())
heat_max = float(heat.max())
heat_norm = (heat - heat_min) / max(heat_max - heat_min, 1e-6)



print(f"sim[{i}, {j}, :, :] shape: {heat.shape}, min={heat_min:.4f}, max={heat_max:.4f}")

plt.imshow(heat, cmap="magma")
plt.colorbar()
plt.show()


from sign_alignment.dift_align import source_foreground_mask
foreground_mask = source_foreground_mask(source_img, source_feature.shape[-2:])
print(f"Foreground mask shape: {foreground_mask.shape}, dtype={foreground_mask.dtype}, unique values={np.unique(foreground_mask)}")

plt.imshow(foreground_mask, cmap="gray")


foreground_mask_tensor = torch.as_tensor(foreground_mask, device=sim_hwhw.device)
masked_sim_hwhw = sim_hwhw * foreground_mask_tensor[..., None, None]
heat_sum = masked_sim_hwhw.sum(axis=(0, 1)).numpy()
print(f"masked sim.sum(axis=(0, 1)) shape: {heat_sum.shape}, min={heat_sum.min():.4f}, max={heat_sum.max():.4f}")
plt.imshow(heat_sum, cmap="magma")
plt.colorbar()
plt.show()

In [ ]:
# dense deformation field
# new method dual softmax

def simply_show(img: np.ndarray, title: str, show_bar: bool, use_log: bool = False):
    if use_log:
        img = np.log(img + 1e-12)
    plt.imshow(img, cmap="magma")
    plt.title(title)
    if show_bar:
        plt.colorbar()
    plt.show()

temperature = 0.03

sim_flat = sim_hwhw.float().reshape(src_h * src_w, dst_h * dst_w)
logits = sim_flat / temperature
simply_show(logits.detach().cpu().numpy(), title="logits", show_bar=True)
soft_match = logits.softmax(dim=1) * logits.softmax(dim=0)
simply_show(soft_match.detach().cpu().numpy(), title="soft match (log)", show_bar=True, use_log=True)
soft_match_hwhw = soft_match.reshape(src_h, src_w, dst_h, dst_w)

# Normalize each source row and take its expected target coordinate.
target_probability = soft_match / soft_match.sum(dim=1, keepdim=True).clamp_min(1e-12)
simply_show(target_probability.detach().cpu().numpy(), title="target probability (log)", show_bar=True, use_log=True)
dst_y, dst_x = torch.meshgrid(
    (torch.arange(dst_h, dtype=sim_flat.dtype) + 0.5) / dst_h,
    (torch.arange(dst_w, dtype=sim_flat.dtype) + 0.5) / dst_w,
    indexing="ij",
)
target_xy = torch.stack([dst_x.reshape(-1), dst_y.reshape(-1)], dim=1)
deformation_field = (target_probability @ target_xy).reshape(src_h, src_w, 2)

src_y, src_x = torch.meshgrid(
    (torch.arange(src_h, dtype=sim_flat.dtype) + 0.5) / src_h,
    (torch.arange(src_w, dtype=sim_flat.dtype) + 0.5) / src_w,
    indexing="ij",
)
source_xy = torch.stack([src_x, src_y], dim=-1)
deformation_flow = deformation_field - source_xy
deformation_magnitude = torch.linalg.vector_norm(deformation_flow, dim=-1)

entropy = -(target_probability * target_probability.clamp_min(1e-12).log()).sum(dim=1)
deformation_certainty = (1.0 - entropy / np.log(dst_h * dst_w)).reshape(src_h, src_w)

foreground = torch.as_tensor(foreground_mask, dtype=torch.bool)
sample = foreground.clone()
sample[1::2, :] = False
sample[:, 1::2] = False

x = source_xy[..., 0][sample].numpy()
y = source_xy[..., 1][sample].numpy()
u = deformation_flow[..., 0][sample].numpy()
v = deformation_flow[..., 1][sample].numpy()
mapped = deformation_field[sample].numpy()
confidence = deformation_certainty[sample].numpy()

fig, axes = plt.subplots(1, 4, figsize=(20, 5), constrained_layout=True)
axes[0].imshow(source_rgb, extent=(0, 1, 1, 0))
axes[0].quiver(
    x, y, u, v, confidence, cmap="viridis",
    angles="xy", scale_units="xy", scale=1, width=0.004,
)
axes[0].set_title("source -> target dense flow")

axes[1].imshow(crop_rgb, extent=(0, 1, 1, 0))
axes[1].scatter(mapped[:, 0], mapped[:, 1], c=confidence, cmap="viridis", s=10)
axes[1].set_title("mapped source foreground on target")

certainty_vis = deformation_certainty.numpy().copy()
certainty_vis[~foreground.numpy()] = np.nan
im = axes[2].imshow(certainty_vis, cmap="viridis", vmin=0, vmax=1)
axes[2].set_title("correspondence certainty")
fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

magnitude_vis = deformation_magnitude.numpy().copy()
magnitude_vis[~foreground.numpy()] = np.nan
im = axes[3].imshow(magnitude_vis, cmap="magma")
axes[3].set_title("normalized flow magnitude")
fig.colorbar(im, ax=axes[3], fraction=0.046, pad=0.04)

for ax in axes[:2]:
    ax.set_xlim(0, 1)
    ax.set_ylim(1, 0)
for ax in axes:
    ax.axis("off")
plt.show()

# Show the deformation vectors directly on the source H x W feature grid.
grid_y, grid_x = torch.meshgrid(
    torch.arange(src_h, dtype=sim_flat.dtype),
    torch.arange(src_w, dtype=sim_flat.dtype),
    indexing="ij",
)
grid_flow_x = deformation_flow[..., 0] * src_w
grid_flow_y = deformation_flow[..., 1] * src_h

quiver_stride = 2
quiver_mask = torch.zeros_like(foreground)
quiver_mask[::quiver_stride, ::quiver_stride] = foreground[
    ::quiver_stride, ::quiver_stride
]

fig, ax = plt.subplots(figsize=(9, 8), constrained_layout=True)
vectors = ax.quiver(
    grid_x[quiver_mask].numpy(),
    grid_y[quiver_mask].numpy(),
    grid_flow_x[quiver_mask].numpy(),
    grid_flow_y[quiver_mask].numpy(),
    deformation_certainty[quiver_mask].numpy(),
    cmap="viridis",
    angles="xy",
    scale_units="xy",
    scale=1,
    width=0.003,
)
ax.set_xlim(-0.5, src_w - 0.5)
ax.set_ylim(src_h - 0.5, -0.5)
ax.set_aspect("equal")
ax.set_xlabel("source feature x (W)")
ax.set_ylabel("source feature y (H)")
ax.set_title(f"dense deformation vectors on {src_h} x {src_w} grid")
ax.grid(alpha=0.2)
fig.colorbar(vectors, ax=ax, label="correspondence certainty")
plt.show()

print(
    f"field={tuple(deformation_field.shape)}, "
    f"foreground mean certainty={deformation_certainty[foreground].mean().item():.4f}, "
    f"foreground mean flow={deformation_magnitude[foreground].mean().item():.4f}"
)


In [ ]:
# soft_match_hwhw visualize [i, j, :, :]
soft_src_i_input = widgets.IntSlider(
    value=src_h // 2, min=0, max=src_h - 1, step=1,
    description="src i", continuous_update=False,
    layout=widgets.Layout(width="420px"),
    style={"description_width": "48px"},
)
soft_src_j_input = widgets.IntSlider(
    value=src_w // 2, min=0, max=src_w - 1, step=1,
    description="src j", continuous_update=False,
    layout=widgets.Layout(width="420px"),
    style={"description_width": "48px"},
)
soft_match_out = widgets.Output()

def _draw_soft_match_slice(i, j):
    with soft_match_out:
        clear_output(wait=True)
        i, j = int(i), int(j)
        heat = soft_match_hwhw[i, j].detach().cpu().numpy()
        heat_min = float(heat.min())
        heat_max = float(heat.max())
        heat_norm = (heat - heat_min) / max(heat_max - heat_min, 1e-12)
        heat_overlay = cv2.resize(
            heat_norm,
            (crop_img_w, crop_img_h),
            interpolation=cv2.INTER_CUBIC,
        )

        src_x_px = (j + 0.5) / src_w * source_img_w
        src_y_px = (i + 0.5) / src_h * source_img_h
        mapped_x = deformation_field[i, j, 0].item() * crop_img_w
        mapped_y = deformation_field[i, j, 1].item() * crop_img_h

        log_heat = np.log(heat + 1e-12)
        fig, axes = plt.subplots(2, 3, figsize=(18, 9), constrained_layout=True)
        axes = axes.ravel()
        axes[0].imshow(source_rgb)
        axes[0].scatter(
            [src_x_px], [src_y_px], c="cyan", s=70,
            edgecolors="black", linewidths=1.2,
        )
        axes[0].set_title(f"prototype/source grid ({i}, {j})")

        axes[1].imshow(crop_rgb)
        axes[1].scatter(
            [mapped_x], [mapped_y], c="cyan", s=70, marker="x",
            linewidths=2.0,
        )
        axes[1].set_title(
            f"target expected point, certainty={deformation_certainty[i, j]:.3f}"
        )

        im = axes[2].imshow(heat, cmap="magma")
        axes[2].set_title(f"soft_match[{i}, {j}, :, :]")
        fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

        im = axes[3].imshow(log_heat, cmap="magma")
        axes[3].set_title(f"log soft_match[{i}, {j}, :, :]")
        fig.colorbar(im, ax=axes[3], fraction=0.046, pad=0.04)

        axes[4].imshow(crop_rgb)
        axes[4].imshow(heat_overlay, cmap="magma", alpha=0.45, vmin=0.0, vmax=1.0)
        axes[4].scatter(
            [mapped_x], [mapped_y], c="cyan", s=70, marker="x",
            linewidths=2.0,
        )
        axes[4].set_title(
            f"target overlay, min={heat_min:.2e}, max={heat_max:.2e}"
        )
        axes[5].axis("off")

        for ax in axes[:5]:
            ax.axis("off")
        display(fig)
        plt.close(fig)

def _redraw_soft_match(_change):
    _draw_soft_match_slice(soft_src_i_input.value, soft_src_j_input.value)

soft_src_i_input.observe(_redraw_soft_match, names="value")
soft_src_j_input.observe(_redraw_soft_match, names="value")
display(widgets.HBox([soft_src_i_input, soft_src_j_input]), soft_match_out)
_draw_soft_match_slice(soft_src_i_input.value, soft_src_j_input.value)


In [ ]:
# create PSR optimizer and plot characteristic loss curves
runner.run([pp.Step("Create PSR optimizer", pp.create_psr_optimizer, pp.vis_psr_optimizer)])

In [ ]:
# run PSR optimization, visualize canonical signs at current boxes, probe current boxes, then continue to final
runner.run([
    pp.Step("Optimize until DIFT probe", pp.optimize_psr_until_dift_probe, pp.vis_optimization),
    # pp.Step("Source sign overlay", pp.create_source_sign_overlay, pp.vis_source_sign_overlay),
    # pp.Step("DIFT affine probe", pp.run_dift_affine_probe, pp.vis_dift_affine_probe),
    pp.Step("Finish PSR optimization", pp.optimize_psr_after_dift_probe, pp.vis_optimization),
])

In [ ]:
# optimization loss history
runner.run([pp.Step("Plot loss history", lambda _: None, pp.vis_loss_history)])

In [ ]:
# 2x2 results comparison: coarse aligned, final optimized, det+final overlay, gt+final overlay
runner.run([pp.Step("Results comparison", lambda _: None, pp.vis_results_comparison)])

In [ ]:
# analyze parameter changes between coarse-aligned and final optimized
runner.run([pp.Step("Parameter changes", lambda _: None, pp.vis_parameter_changes)])

In [ ]:
import numpy as np
import torch

a_np = np.array([[1, 2, 3],
                 [4, 5, 6],
                 [7, 8, 9]], dtype=np.float32)

b_np = np.array([[9, 8, 7],
                 [6, 5, 4],
                 [3, 2, 1]], dtype=np.float32)


a = torch.tensor(a_np)
b = torch.tensor(b_np)

print(a)
print(b)

c = a.shape[0]
print(c)
a_flat = a.reshape(c,-1)
b_flat = b.reshape(c,-1)

print(a_flat)
print(b_flat)

sim_cosine = torch.nn.functional.cosine_similarity(a_flat, b_flat, dim=1)
sim_at = a_flat @ b_flat.T

print(sim_cosine)
print(sim_at)